# NB01b — Préparation VHH (couches L1–L6) sur Colab
**Projet 1.1 — nouvelles données OAS-nano (Wang 2024, alpaga)**

Ce notebook lance `scripts/prepare_vhh.py` (L1 gaps · L2 alphabet · L3 flags OAS · L4 longueur · L5 intégrité des régions · L6 cystéines) sur les fichiers bruts du Drive.

| | Chemin (Drive `AI_Project_Shared`) |
|---|---|
| Entrée | `data/raw/<fichiers .csv.gz>` |
| Sortie test | `data/processed/v2_test/` (50 000 lignes/fichier, jetable) |
| Sortie finale | `data/processed/v2/` (`vhh_clean.csv`, `vhh_clean_meta.csv`, `cleaning_report.csv`, `length_histogram_before_length_filter.csv`, `manifest.json`) |

Ordre : cellule 1 (réglages + bootstrap) → 2 (fichiers) → 3 (`--inspect`) → 4 (test 50 000) → **lire le rapport** → 5 (run complet) → 6 (résultats).
Après L6, `prepare_vhh.py` fait une **déduplication exacte** (étape de sortie, pas une couche) : `vhh_clean.csv` est donc prêt pour NB02, et `n_occurrences` garde le nombre de copies.

In [ ]:
# ==============================================================================
# Cellule 1 : RÉGLAGES + bootstrap Colab (Drive, clone du dépôt)
# ==============================================================================
# ------------------------------ RÉGLAGES -------------------------------------
RAW_FILES = [                       # noms exacts dans AI_Project_Shared/data/raw/
    "SRR30246849_1_Heavy_IGHG2B.csv.gz",
    "SRR30246855_1_Heavy_IGHG2C.csv.gz",
    "SRR30246845_1_Heavy_IGHG2B.csv.gz",
    "SRR30246862_1_Heavy_IGHG2C.csv.gz",
]
FINAL_VERSION = "v2"                # dossier de sortie final
TEST_VERSION = "v2_test"            # dossier du test (jetable)
TEST_ROWS = 50_000                  # lignes lues par fichier pour le test
OVERWRITE = False                   # True seulement pour écraser volontairement v2
BRANCH = "main"
# ------------------------------------------------------------------------------
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/AI_Project_Shared")
if not DRIVE.exists():
    raise FileNotFoundError(f"{DRIVE} introuvable : ajoutez le raccourci dans 'Mon Drive'.")
os.environ["AI_PROJECT_DIR"] = str(DRIVE)

REPO = Path("/content/Recherche_AI")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

SCRIPT = REPO / "scripts" / "prepare_vhh.py"
if not SCRIPT.exists():
    raise FileNotFoundError("scripts/prepare_vhh.py absent de la branche : faites git add / commit / push avant.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
from utils import get_paths, MIN_LEN, MAX_LEN
PATHS = get_paths()
RAW_DIR, PROCESSED = PATHS["raw"], PATHS["processed"]
print("Raw       :", RAW_DIR)
print("Processed :", PROCESSED)
print(f"Bornes de longueur (utils.py) : {MIN_LEN}-{MAX_LEN}")
print("Commit    :", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

def run_prepare(version, extra_args=()):
    """Lance prepare_vhh.py avec DATA_VERSION=version et affiche la sortie en direct."""
    env = {**os.environ, "DATA_VERSION": version}
    cmd = [sys.executable, str(SCRIPT), "--inputs", *INPUTS, *extra_args]
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(REPO))
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"prepare_vhh.py a échoué (code {p.returncode}) : lisez le message ci-dessus.")


In [ ]:
# ==============================================================================
# Cellule 2 : Vérification des fichiers bruts
# ==============================================================================
INPUTS = []
missing = []
for name in RAW_FILES:
    p = RAW_DIR / name
    if p.exists():
        INPUTS.append(str(p)); print(f"OK   {name}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        missing.append(name); print(f"ABSENT {name}")
print("\nAutres fichiers dans raw/ (non utilisés) :",
      sorted(q.name for q in RAW_DIR.glob("*") if q.name not in RAW_FILES))
if missing:
    raise FileNotFoundError(f"Fichiers manquants dans {RAW_DIR} : {missing}")

if (PROCESSED / FINAL_VERSION / "manifest.json").exists() and not OVERWRITE:
    print(f"\nATTENTION : {FINAL_VERSION} existe déjà. Le run complet (cellule 5) s'arrêtera sauf si OVERWRITE = True.")


In [ ]:
# ==============================================================================
# Cellule 3 : --inspect (colonnes présentes dans chaque fichier)
# ==============================================================================
run_prepare(TEST_VERSION, ["--inspect"])


**À lire dans la sortie `--inspect`** : les 7 colonnes de régions (`fwr1_aa` … `fwr4_aa`) et les 4 flags (`productive`, `stop_codon`, `vj_in_frame`, `v_frameshift`) sont-ils présents ? Si une région manque, L5 est ignorée ; si une des 5 régions de L6 manque, L6 est ignorée (message `NOTE`).

In [ ]:
# ==============================================================================
# Cellule 4 : Test rapide (50 000 lignes par fichier) -> data/processed/v2_test
# ==============================================================================
run_prepare(TEST_VERSION, ["--max-rows", str(TEST_ROWS)])

import pandas as pd
pd.set_option("display.width", 200)
test_dir = PROCESSED / TEST_VERSION
rep = pd.read_csv(test_dir / "cleaning_report.csv")
rep["% retiré"] = (100 * rep["removed"] / rep["rows_before"].clip(lower=1)).round(2)
display(rep)
print("Notes du manifest :")
for n in json.load(open(test_dir / "manifest.json", encoding="utf-8"))["notes"]:
    print(" -", n)

# Disposition des régions : L6 ne vérifie que les lignes 25/8/17/38
m = pd.read_csv(test_dir / "vhh_clean_meta.csv", dtype=str, keep_default_na=False,
                usecols=["fwr1_aa", "cdr1_aa", "fwr2_aa", "fwr3_aa"])
for c in m.columns:
    print(c, "longueurs les plus fréquentes :", m[c].str.len().value_counts().head(4).to_dict())


**Avant le run complet** : regardez `% retiré` par couche. Une couche qui retire une très grande part (surtout L5 ou L6) signale une règle ou un format de colonnes inadapté, pas forcément de mauvaises données. Regardez la note `L6 coverage` : si le pourcentage de lignes vérifiées est faible, les régions OAS-nano n'ont pas la disposition 25/8/17/38 (voir les longueurs fréquentes affichées) et L6 ne filtre presque rien : il faudra adapter la règle, pas faire confiance à L6.

In [ ]:
# ==============================================================================
# Cellule 5 : Run complet -> data/processed/v2
# ==============================================================================
RUN_FULL = True      # mettre False pour ne lancer que le test
final_dir = PROCESSED / FINAL_VERSION
if RUN_FULL:
    if (final_dir / "manifest.json").exists() and not OVERWRITE:
        raise FileExistsError(f"{final_dir} existe déjà. Choisissez FINAL_VERSION='v3' ou OVERWRITE=True (jamais écraser v1).")
    run_prepare(FINAL_VERSION)


In [ ]:
# ==============================================================================
# Cellule 6 : Résultats (rapport, histogramme des longueurs, aperçu)
# ==============================================================================
import matplotlib.pyplot as plt
rep = pd.read_csv(final_dir / "cleaning_report.csv")
rep["% retiré"] = (100 * rep["removed"] / rep["rows_before"].clip(lower=1)).round(2)
display(rep)

man = json.load(open(final_dir / "manifest.json", encoding="utf-8"))
print("Commit :", man["git_commit"], "| version :", man["data_version"], "| dédup :", man["parameters"]["deduplication"])
for n in man["notes"]:
    print("NOTE:", n)

h = pd.read_csv(final_dir / "length_histogram_before_length_filter.csv")
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(h["length"], h["n_sequences"], width=1.0, color="steelblue")
ax.axvline(MIN_LEN, color="red", ls="--"); ax.axvline(MAX_LEN, color="orange", ls="--")
ax.set_xlabel("Longueur (avant filtre L4)"); ax.set_ylabel("Séquences"); ax.set_title("Distribution des longueurs")
plt.tight_layout(); plt.show()

meta = pd.read_csv(final_dir / "vhh_clean_meta.csv", dtype=str, keep_default_na=False, nrows=5)
display(meta)

n_total = sum(1 for _ in open(final_dir / "vhh_clean.csv", encoding="utf-8")) - 1
n_unique = pd.read_csv(final_dir / "vhh_clean.csv", dtype=str)["sequence"].nunique()
print(f"\nvhh_clean.csv : {n_total:,} lignes, {n_unique:,} séquences uniques")
assert n_unique == n_total, "doublons exacts dans vhh_clean.csv : NB02 refusera ce fichier"
print("OK : aucun doublon exact, vhh_clean.csv est prêt pour NB02.")
